# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates a workflow for loading and exploring the FAIRˆ<sup>2</sup> dataset using the `mlcroissant` library. The dataset provides mass spectrometry results of protein abundance, modifications, and sequences in extracellular vesicles from human mast cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL, enabling automated metadata discovery and easy access via Python.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(url)
# The dataset.metadata field is an object, not a dictionary
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available Record Sets and their fields with their `@id`, `name`, and `description` using the Croissant metadata structure.

In [ ]:
# Print out available record sets and their fields with @id
print('Record Sets in this dataset:')
record_sets = list(dataset.record_sets)
if not record_sets:
    print('No record sets found. Please check the schema for more information.')
else:
    for recset in record_sets:
        print(f"- Record Set @id: {recset['@id']}")
        print(f"  name: {recset.get('name', 'N/A')}")
        print(f"  description: {recset.get('description', 'N/A')}")
        print(f"  Fields:")
        for field in recset['fields']:
            print(f"    - Field @id: {field['@id']} | name: {field.get('name', 'N/A')} | description: {field.get('description', 'N/A')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their unique `@id`.

In [ ]:
# Extract all record sets as DataFrames using their @id
dataframes = {}
record_set_ids = []
for recset in dataset.record_sets:
    rec_id = recset['@id']
    record_set_ids.append(rec_id)
    print(f"Loading records from record set @id: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Rows: {len(df)}")
    if len(df) > 0:
        print(df.head(2))
    print()
# For further analysis, select the first non-empty record set
main_record_set_id = next((rsid for rsid in record_set_ids if not dataframes[rsid].empty), None)
if main_record_set_id:
    print(f"Chosen record set for analysis: {main_record_set_id}")
    print(f"Fields/Columns: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping. All operations reference fields by their `@id`.

In [ ]:
# --- Select a numeric field and a group field for demonstration (by their @id) ---
df = dataframes[main_record_set_id]

# Print out fields for easy selection
print('Available fields for this record set:')
print(df.columns.tolist())

# Choose a numeric field and a group/categorical field by their @id. You may adjust this according to the actual field IDs:
# Example placeholder: replace with actual field @ids from above output
numeric_field_id = None
for col in df.columns:
    if df[col].dtype in ['int64','float64'] and not numeric_field_id:
        numeric_field_id = col
group_field_id = None
for col in df.columns:
    if df[col].dtype == object and col != numeric_field_id:
        group_field_id = col
        break
print(f"Using numeric field: {numeric_field_id}, and group field: {group_field_id}")

# Threshold for filtering
if numeric_field_id is not None and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    
    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst 5 normalized values of '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Group by a categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped.head())
else:
    print('No numeric field available for analysis in this record set.')

## 5. Visualization
Plot distributions or relationships between fields. Using `matplotlib` and `seaborn` for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# If group field is available, plot grouped means
if group_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(10,4))
    group_data = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    group_data.plot(kind='bar')
    plt.title(f'Mean {numeric_field_id} grouped by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've loaded, examined, and visualized the mass spectrometry dataset using the `mlcroissant` library. Key findings and further analyses can be guided by your research questions and the provided data schema. For more advanced analysis, consider exploring other fields, filtering strategies, and modeling approaches.